<a href="https://colab.research.google.com/github/shouryakumar6290/AI-Road-Accident-Severity-Project/blob/main/AI_Road_Accident_Severity_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pandas scikit-learn matplotlib seaborn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


In [ ]:
df = pd.read_csv("UK_Accident.csv")
df.head()


In [ ]:
df = df.dropna()
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day
df['Hour'] = df['Time'].str.split(':').str[0].astype(float)
df = df.drop(['Date','Time','Accident_Index','Unnamed: 0'], axis=1)


In [ ]:
for col in df.select_dtypes(include=["object"]).columns:
    df[col] = df[col].astype("category").cat.codes


In [ ]:
X = df.drop("Accident_Severity", axis=1)
y = df["Accident_Severity"]


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [ ]:
param_grid = {
    'n_estimators':[100,200],
    'max_depth':[10,20,None],
    'min_samples_split':[2,5],
    'min_samples_leaf':[1,2]
}

rf = RandomForestClassifier(random_state=42)
grid = GridSearchCV(rf, param_grid, cv=3, n_jobs=-1)
grid.fit(X_train, y_train)
model = grid.best_estimator_


In [ ]:
y_pred = model.predict(X_test)
print("Best Parameters:", grid.best_params_)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))


In [ ]:
importances = model.feature_importances_
feat_names = X.columns
feat_imp = pd.Series(importances, index=feat_names).sort_values(ascending=False)
plt.figure(figsize=(10,6))
sns.barplot(x=feat_imp.values, y=feat_imp.index)
plt.title("Feature Importance")
plt.show()


In [ ]:
sns.countplot(x="Accident_Severity", data=df)
plt.title("Accident Severity Distribution")
plt.show()


In [ ]:
sample_input = {}
for col in X.columns:
    val = input(f"Enter value for {col}: ")
    try:
        sample_input[col] = float(val)
    except:
        sample_input[col] = int(val)

sample_df = pd.DataFrame([sample_input])
sample_scaled = scaler.transform(sample_df)
pred = model.predict(sample_scaled)
print("Predicted Accident Severity:", pred[0])


In [ ]:
!pip install ipywidgets
from ipywidgets import interact_manual, Dropdown, IntText, FloatText
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler


In [ ]:
weather_options = {
    "Clear": 0,
    "Rainy": 1,
    "Snow": 2,
    "Fog": 3,
    "Other": 4
}

light_options = {
    "Daylight": 0,
    "Darkness – lights lit": 1,
    "Darkness – lights unlit": 2,
    "Other": 3
}

road_type_options = {
    "Roundabout":0,
    "One way street":1,
    "Dual carriageway":2,
    "Single carriageway":3,
    "Slip road":4,
    "Other":5
}

severity_mapping = {
    1:"Slight",
    2:"Serious",
    3:"Fatal"
}


In [ ]:
def predict_accident(
    Longitude: FloatText,
    Latitude: FloatText,
    Number_of_Vehicles: IntText,
    Number_of_Casualties: IntText,
    Day_of_Week: Dropdown,
    Weather: Dropdown,
    Light_Conditions: Dropdown,
    Road_Type: Dropdown,
    Speed_limit: IntText
):
    sample = pd.DataFrame({
        'Longitude':[Longitude],
        'Latitude':[Latitude],
        'Number_of_Vehicles':[Number_of_Vehicles],
        'Number_of_Casualties':[Number_of_Casualties],
        'Day_of_Week':[Day_of_Week],
        'Weather_Conditions':[weather_options[Weather]],
        'Light_Conditions':[light_options[Light_Conditions]],
        'Road_Type':[road_type_options[Road_Type]],
        'Speed_limit':[Speed_limit]
    })

    sample_scaled = scaler.transform(sample)
    pred = model.predict(sample_scaled)[0]
    print(f"✅ Predicted Accident Severity: {severity_mapping.get(pred, 'Unknown')}")


In [ ]:

sample_template = pd.DataFrame(columns=X.columns)


for col in X.columns:
    sample_template[col] = [0]


In [ ]:
def predict_accident_friendly(
    Longitude: float,
    Latitude: float,
    Number_of_Vehicles: int,
    Number_of_Casualties: int,
    Day_of_Week: int,
    Weather: str,
    Light_Conditions: str,
    Road_Type: str,
    Speed_limit: int
):
    sample = sample_template.copy()

    sample['Longitude'] = Longitude
    sample['Latitude'] = Latitude
    sample['Number_of_Vehicles'] = Number_of_Vehicles
    sample['Number_of_Casualties'] = Number_of_Casualties
    sample['Day_of_Week'] = Day_of_Week
    sample['Weather_Conditions'] = weather_options[Weather]
    sample['Light_Conditions'] = light_options[Light_Conditions]
    sample['Road_Type'] = road_type_options[Road_Type]
    sample['Speed_limit'] = Speed_limit

    sample_scaled = scaler.transform(sample)
    pred = model.predict(sample_scaled)[0]

    print(f"✅ Predicted Accident Severity: {severity_mapping.get(pred, 'Unknown')}")


In [ ]:
from ipywidgets import interact_manual, FloatText, IntText, Dropdown

interact_manual(
    predict_accident_friendly,
    Longitude=FloatText(value=-0.1257, description="Longitude"),
    Latitude=FloatText(value=51.5085, description="Latitude"),
    Number_of_Vehicles=IntText(value=2, description="Vehicles"),
    Number_of_Casualties=IntText(value=1, description="Casualties"),
    Day_of_Week=Dropdown(options=[1,2,3,4,5,6,7], description="Day (Mon=1)"),
    Weather=Dropdown(options=list(weather_options.keys()), description="Weather"),
    Light_Conditions=Dropdown(options=list(light_options.keys()), description="Light"),
    Road_Type=Dropdown(options=list(road_type_options.keys()), description="Road Type"),
    Speed_limit=IntText(value=30, description="Speed Limit")
)


In [ ]:
from ipywidgets import interact_manual, FloatText, IntText, Dropdown

interact_manual(
    predict_accident_friendly,
    Longitude=FloatText(value=-0.1257, description="Longitude"),
    Latitude=FloatText(value=51.5085, description="Latitude"),
    Number_of_Vehicles=IntText(value=2, description="Vehicles"),
    Number_of_Casualties=IntText(value=1, description="Casualties"),
    Day_of_Week=Dropdown(options=[1,2,3,4,5,6,7], description="Day (Mon=1)"),
    Weather=Dropdown(options=list(weather_options.keys()), description="Weather"),
    Light_Conditions=Dropdown(options=list(light_options.keys()), description="Light"),
    Road_Type=Dropdown(options=list(road_type_options.keys()), description="Road Type"),
    Speed_limit=IntText(value=30, description="Speed Limit")
)

In [ ]:
from ipywidgets import interact_manual, FloatText, IntText, Dropdown

interact_manual(
    predict_accident_friendly,
    Longitude=FloatText(value=-0.1257, description="Longitude"),
    Latitude=FloatText(value=51.5085, description="Latitude"),
    Number_of_Vehicles=IntText(value=2, description="Vehicles"),
    Number_of_Casualties=IntText(value=1, description="Casualties"),
    Day_of_Week=Dropdown(options=[1,2,3,4,5,6,7], description="Day (Mon=1)"),
    Weather=Dropdown(options=list(weather_options.keys()), description="Weather"),
    Light_Conditions=Dropdown(options=list(light_options.keys()), description="Light"),
    Road_Type=Dropdown(options=list(road_type_options.keys()), description="Road Type"),
    Speed_limit=IntText(value=30, description="Speed Limit")
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x='Accident_Severity', data=df, palette='coolwarm')
plt.title('Distribution of Accident Severity')
plt.xlabel('Severity (1=Slight, 2=Serious, 3=Fatal)')
plt.ylabel('Number of Accidents')
plt.show()


In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(x='Weather_Conditions', hue='Accident_Severity', data=df, palette='Set2')
plt.title('Accident Severity by Weather')
plt.xlabel('Weather Condition (0=Clear, 1=Rainy, 2=Snow, etc.)')
plt.ylabel('Number of Accidents')
plt.legend(title='Severity')
plt.show()


In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(x='Road_Type', hue='Accident_Severity', data=df, palette='Set1')
plt.title('Accident Severity by Road Type')
plt.xlabel('Road Type Code')
plt.ylabel('Number of Accidents')
plt.legend(title='Severity')
plt.show()


In [ ]:
plt.figure(figsize=(10,5))
sns.boxplot(x='Hour', y='Accident_Severity', data=df, palette='cool')
plt.title('Accident Severity by Hour of Day')
plt.xlabel('Hour')
plt.ylabel('Severity')
plt.show()


In [ ]:
importances = model.feature_importances_
feat_names = X.columns
feat_imp = pd.Series(importances, index=feat_names).sort_values(ascending=False)

plt.figure(figsize=(10,6))
sns.barplot(x=feat_imp.values, y=feat_imp.index, palette='viridis')
plt.title('Feature Importance: Factors Leading to Severe Accidents')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.show()


In [ ]:
!pip install ipywidgets matplotlib seaborn
import pandas as pd
import numpy as np
from ipywidgets import interact_manual, FloatText, IntText, Dropdown
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler


In [ ]:
weather_options = {"Clear":0, "Rainy":1, "Snow":2, "Fog":3, "Other":4}
light_options = {"Daylight":0, "Dark – lights lit":1, "Dark – lights unlit":2, "Other":3}
road_type_options = {"Roundabout":0,"One way":1,"Dual carriageway":2,"Single carriageway":3,"Slip road":4,"Other":5}
severity_mapping = {1:"Slight", 2:"Serious", 3:"Fatal"}


sample_template = pd.DataFrame(columns=X.columns)
for col in X.columns:
    sample_template[col] = [0]


In [ ]:
def predict_and_visualize(
    Longitude: float,
    Latitude: float,
    Number_of_Vehicles: int,
    Number_of_Casualties: int,
    Day_of_Week: int,
    Weather: str,
    Light_Conditions: str,
    Road_Type: str,
    Speed_limit: int
):

    sample = sample_template.copy()
    sample['Longitude'] = Longitude
    sample['Latitude'] = Latitude
    sample['Number_of_Vehicles'] = Number_of_Vehicles
    sample['Number_of_Casualties'] = Number_of_Casualties
    sample['Day_of_Week'] = Day_of_Week
    sample['Weather_Conditions'] = weather_options[Weather]
    sample['Light_Conditions'] = light_options[Light_Conditions]
    sample['Road_Type'] = road_type_options[Road_Type]
    sample['Speed_limit'] = Speed_limit


    sample_scaled = scaler.transform(sample)
    pred = model.predict(sample_scaled)[0]

    print(f"\n✅ Predicted Accident Severity: {severity_mapping[pred]}\n")


    plt.figure(figsize=(6,4))
    sns.countplot(x='Accident_Severity', data=df, palette='coolwarm')
    plt.title('Distribution of Accident Severity')
    plt.show()

    plt.figure(figsize=(8,5))
    sns.countplot(x='Weather_Conditions', hue='Accident_Severity', data=df, palette='Set2')
    plt.title('Accident Severity by Weather')
    plt.show()

    plt.figure(figsize=(8,5))
    sns.countplot(x='Road_Type', hue='Accident_Severity', data=df, palette='Set1')
    plt.title('Accident Severity by Road Type')
    plt.show()

    plt.figure(figsize=(10,5))
    sns.boxplot(x='Hour', y='Accident_Severity', data=df, palette='cool')
    plt.title('Accident Severity by Hour')
    plt.show()

    importances = model.feature_importances_
    feat_imp = pd.Series(importances, index=X.columns).sort_values(ascending=False)
    plt.figure(figsize=(10,6))
    sns.barplot(x=feat_imp.values, y=feat_imp.index, palette='viridis')
    plt.title('Feature Importance: Factors Leading to Severe Accidents')
    plt.show()


In [ ]:
interact_manual(
    predict_and_visualize,
    Longitude=FloatText(value=-0.1257, description="Longitude"),
    Latitude=FloatText(value=51.5085, description="Latitude"),
    Number_of_Vehicles=IntText(value=2, description="Vehicles"),
    Number_of_Casualties=IntText(value=1, description="Casualties"),
    Day_of_Week=Dropdown(options=[1,2,3,4,5,6,7], description="Day (Mon=1)"),
    Weather=Dropdown(options=list(weather_options.keys()), description="Weather"),
    Light_Conditions=Dropdown(options=list(light_options.keys()), description="Light"),
    Road_Type=Dropdown(options=list(road_type_options.keys()), description="Road Type"),
    Speed_limit=IntText(value=30, description="Speed Limit")
)


In [ ]:
def give_suggestions(sample_row):
    suggestions = []

    if sample_row['Weather_Conditions'].iloc[0] == weather_options["Rainy"]:
        suggestions.append("Drive slowly and maintain a safe distance; roads may be slippery.")
    elif sample_row['Weather_Conditions'].iloc[0] == weather_options["Snow"]:
        suggestions.append("Use snow chains or winter tires; drive very cautiously.")


    if sample_row['Light_Conditions'].iloc[0] != light_options["Daylight"]:
        suggestions.append("Ensure headlights are on and reduce speed at night or in low visibility.")


    if sample_row['Speed_limit'].iloc[0] > 40:
        suggestions.append("Consider reducing speed; higher speed increases accident severity.")


    if sample_row['Number_of_Vehicles'].iloc[0] > 2:
        suggestions.append("Be extra cautious; multi-vehicle situations are riskier.")

    if sample_row['Road_Type'].iloc[0] in [road_type_options["Roundabout"], road_type_options["Dual carriageway"]]:
        suggestions.append("Pay attention to traffic rules; intersections and dual carriageways are high risk.")

    if not suggestions:
        suggestions.append("Maintain normal caution while driving; conditions seem safe.")

    print("\n💡 Safety Suggestions to Avoid Accidents:")
    for s in suggestions:
        print("-", s)


In [ ]:
interact_manual(
    predict_and_visualize,
    Longitude=FloatText(value=-0.1257, description="Longitude"),
    Latitude=FloatText(value=51.5085, description="Latitude"),
    Number_of_Vehicles=IntText(value=2, description="Vehicles"),
    Number_of_Casualties=IntText(value=1, description="Casualties"),
    Day_of_Week=Dropdown(options=[1,2,3,4,5,6,7], description="Day (Mon=1)"),
    Weather=Dropdown(options=list(weather_options.keys()), description="Weather"),
    Light_Conditions=Dropdown(options=list(light_options.keys()), description="Light"),
    Road_Type=Dropdown(options=list(road_type_options.keys()), description="Road Type"),
    Speed_limit=IntText(value=30, description="Speed Limit")
)

In [ ]:
from ipywidgets import interact_manual, FloatText, IntText, Dropdown

interact_manual(
    predict_and_visualize,
    Longitude=FloatText(value=-0.1257, description="Longitude"),
    Latitude=FloatText(value=51.5085, description="Latitude"),
    Number_of_Vehicles=IntText(value=2, description="Vehicles"),
    Number_of_Casualties=IntText(value=1, description="Casualties"),
    Day_of_Week=Dropdown(
        options=[
            ( "Monday", 1 ),
            ( "Tuesday", 2 ),
            ( "Wednesday", 3 ),
            ( "Thursday", 4 ),
            ( "Friday", 5 ),
            ( "Saturday", 6 ),
            ( "Sunday", 7 )
        ],
        description="Day"
    ),
    Weather=Dropdown(
        options=[ "Clear", "Rainy", "Snow", "Fog", "Other" ],
        description="Weather"
    ),
    Light_Conditions=Dropdown(
        options=[ "Daylight", "Dark – lights lit", "Dark – lights unlit", "Other" ],
        description="Light"
    ),
    Road_Type=Dropdown(
        options=[ "Roundabout", "One way", "Dual carriageway", "Single carriageway", "Slip road", "Other" ],
        description="Road Type"
    ),
    Speed_limit=IntText(value=30, description="Speed Limit")
)


In [ ]:

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ipywidgets import interact_manual, FloatText, IntText, Dropdown


severity_labels = {1: "Slight", 2: "Serious", 3: "Fatal"}
weather_labels = {0: "Clear", 1: "Rainy", 2: "Snow", 3: "Fog", 4: "Other"}
light_labels = {0: "Daylight", 1: "Dark – lights lit", 2: "Dark – lights unlit", 3: "Other"}
road_labels = {0:"Roundabout",1:"One way",2:"Dual carriageway",3:"Single carriageway",4:"Slip road",5:"Other"}


df = pd.read_csv("UK_Accident.csv")
df['Accident_Severity_Label'] = df['Accident_Severity'].map(severity_labels)
df['Weather_Label'] = df['Weather_Conditions'].map(weather_labels)
df['Light_Label'] = df['Light_Conditions'].map(light_labels)
df['Road_Label'] = df['Road_Type'].map(road_labels)

weather_options = list(weather_labels.values())
light_options = list(light_labels.values())
road_type_options = list(road_labels.values())


def predict_and_visualize(
    Longitude: float,
    Latitude: float,
    Number_of_Vehicles: int,
    Number_of_Casualties: int,
    Day_of_Week: int,
    Weather: str,
    Light_Conditions: str,
    Road_Type: str,
    Speed_limit: int
):

    sample = pd.DataFrame(columns=X.columns)
    sample.loc[0] = 0
    sample['Longitude'] = Longitude
    sample['Latitude'] = Latitude
    sample['Number_of_Vehicles'] = Number_of_Vehicles
    sample['Number_of_Casualties'] = Number_of_Casualties
    sample['Day_of_Week'] = Day_of_Week
    sample['Weather_Conditions'] = list(weather_labels.keys())[weather_options.index(Weather)]
    sample['Light_Conditions'] = list(light_labels.keys())[light_options.index(Light_Conditions)]
    sample['Road_Type'] = list(road_labels.keys())[road_type_options.index(Road_Type)]
    sample['Speed_limit'] = Speed_limit


    sample_scaled = scaler.transform(sample)
    pred = model.predict(sample_scaled)[0]

    print(f"\n✅ Predicted Accident Severity: {severity_labels[pred]}\n")


    plt.figure(figsize=(6,4))
    sns.countplot(x='Accident_Severity_Label', data=df, palette='coolwarm')
    plt.title('Distribution of Accident Severity')
    plt.xlabel('Severity')
    plt.ylabel('Number of Accidents')
    plt.show()

    plt.figure(figsize=(8,5))
    sns.countplot(x='Weather_Label', hue='Accident_Severity_Label', data=df, palette='Set2')
    plt.title('Accident Severity by Weather')
    plt.xlabel('Weather')
    plt.ylabel('Number of Accidents')
    plt.legend(title='Severity')
    plt.show()

    plt.figure(figsize=(8,5))
    sns.countplot(x='Road_Label', hue='Accident_Severity_Label', data=df, palette='Set1')
    plt.title('Accident Severity by Road Type')
    plt.xlabel('Road Type')
    plt.ylabel('Number of Accidents')
    plt.legend(title='Severity')
    plt.show()

    plt.figure(figsize=(10,5))
    sns.boxplot(x='Hour', y='Accident_Severity', data=df, palette='cool')
    plt.title('Accident Severity by Hour')
    plt.xlabel('Hour of Day')
    plt.ylabel('Severity')
    plt.show()

    importances = model.feature_importances_
    feat_imp = pd.Series(importances, index=X.columns).sort_values(ascending=False)
    plt.figure(figsize=(10,6))
    sns.barplot(x=feat_imp.values, y=feat_imp.index, palette='viridis')
    plt.title('Feature Importance: Factors Leading to Severe Accidents')
    plt.show()

    suggestions = []
    if sample['Weather_Conditions'].iloc[0] == 1:
        suggestions.append("Drive slowly and keep safe distance; rainy roads are slippery.")
    if sample['Weather_Conditions'].iloc[0] == 2:
        suggestions.append("Use snow tires and drive cautiously; snow increases risk.")
    if sample['Light_Conditions'].iloc[0] != 0:
        suggestions.append("Ensure headlights are on and reduce speed at night or low visibility.")
    if sample['Speed_limit'].iloc[0] > 40:
        suggestions.append("Reduce speed; higher speed increases accident severity.")
    if sample['Number_of_Vehicles'].iloc[0] > 2:
        suggestions.append("Be extra careful; multi-vehicle situations are riskier.")
    if sample['Road_Type'].iloc[0] in [0,2]:
        suggestions.append("Pay attention to traffic rules; intersections and dual carriageways are high risk.")
    if not suggestions:
        suggestions.append("Maintain normal caution; conditions seem safe.")

    print("\n💡 Safety Suggestions to Avoid Accidents:")
    for s in suggestions:
        print("-", s)


In [ ]:

df['Hour'] = pd.to_datetime(df['Time'], format='%H:%M', errors='coerce').dt.hour


In [ ]:
plt.figure(figsize=(10,5))
sns.boxplot(x='Hour', y='Accident_Severity', data=df, palette='cool')
plt.title('Accident Severity by Hour')
plt.xlabel('Hour of Day')
plt.ylabel('Severity')
plt.show()


In [ ]:

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ipywidgets import interact_manual, FloatText, IntText, Dropdown


df = pd.read_csv("UK_Accident.csv")

severity_labels = {1: "Slight", 2: "Serious", 3: "Fatal"}
weather_labels = {0: "Clear", 1: "Rainy", 2: "Snow", 3: "Fog", 4: "Other"}
light_labels = {0: "Daylight", 1: "Dark – lights lit", 2: "Dark – lights unlit", 3: "Other"}
road_labels = {0:"Roundabout",1:"One way",2:"Dual carriageway",3:"Single carriageway",4:"Slip road",5:"Other"}

df['Accident_Severity_Label'] = df['Accident_Severity'].map(severity_labels)
df['Weather_Label'] = df['Weather_Conditions'].map(weather_labels)
df['Light_Label'] = df['Light_Conditions'].map(light_labels)
df['Road_Label'] = df['Road_Type'].map(road_labels)

df['Hour'] = pd.to_datetime(df['Time'], format='%H:%M', errors='coerce').dt.hour

weather_options = list(weather_labels.values())
light_options = list(light_labels.values())
road_type_options = list(road_labels.values())

def predict_and_visualize(
    Longitude: float,
    Latitude: float,
    Number_of_Vehicles: int,
    Number_of_Casualties: int,
    Day_of_Week: int,
    Weather: str,
    Light_Conditions: str,
    Road_Type: str,
    Speed_limit: int
):

    sample = pd.DataFrame(columns=X.columns)
    sample.loc[0] = 0
    sample['Longitude'] = Longitude
    sample['Latitude'] = Latitude
    sample['Number_of_Vehicles'] = Number_of_Vehicles
    sample['Number_of_Casualties'] = Number_of_Casualties
    sample['Day_of_Week'] = Day_of_Week
    sample['Weather_Conditions'] = list(weather_labels.keys())[weather_options.index(Weather)]
    sample['Light_Conditions'] = list(light_labels.keys())[light_options.index(Light_Conditions)]
    sample['Road_Type'] = list(road_labels.keys())[road_type_options.index(Road_Type)]
    sample['Speed_limit'] = Speed_limit


    sample_scaled = scaler.transform(sample)
    pred = model.predict(sample_scaled)[0]

    print(f"\n✅ Predicted Accident Severity: {severity_labels[pred]}\n")


    plt.figure(figsize=(6,4))
    sns.countplot(x='Accident_Severity_Label', data=df, palette='coolwarm')
    plt.title('Distribution of Accident Severity')
    plt.xlabel('Severity')
    plt.ylabel('Number of Accidents')
    plt.show()

    plt.figure(figsize=(8,5))
    sns.countplot(x='Weather_Label', hue='Accident_Severity_Label', data=df, palette='Set2')
    plt.title('Accident Severity by Weather')
    plt.xlabel('Weather')
    plt.ylabel('Number of Accidents')
    plt.legend(title='Severity')
    plt.show()

    plt.figure(figsize=(8,5))
    sns.countplot(x='Road_Label', hue='Accident_Severity_Label', data=df, palette='Set1')
    plt.title('Accident Severity by Road Type')
    plt.xlabel('Road Type')
    plt.ylabel('Number of Accidents')
    plt.legend(title='Severity')
    plt.show()

    plt.figure(figsize=(10,5))
    sns.boxplot(x='Hour', y='Accident_Severity', data=df, palette='cool')
    plt.title('Accident Severity by Hour')
    plt.xlabel('Hour of Day')
    plt.ylabel('Severity')
    plt.show()


    importances = model.feature_importances_
    feat_imp = pd.Series(importances, index=X.columns).sort_values(ascending=False)
    plt.figure(figsize=(10,6))
    sns.barplot(x=feat_imp.values, y=feat_imp.index, palette='viridis')
    plt.title('Feature Importance: Factors Leading to Severe Accidents')
    plt.show()


    suggestions = []
    if sample['Weather_Conditions'].iloc[0] == 1:
        suggestions.append("Drive slowly and keep safe distance; rainy roads are slippery.")
    if sample['Weather_Conditions'].iloc[0] == 2:
        suggestions.append("Use snow tires and drive cautiously; snow increases risk.")
    if sample['Light_Conditions'].iloc[0] != 0:
        suggestions.append("Ensure headlights are on and reduce speed at night or low visibility.")
    if sample['Speed_limit'].iloc[0] > 40:
        suggestions.append("Reduce speed; higher speed increases accident severity.")
    if sample['Number_of_Vehicles'].iloc[0] > 2:
        suggestions.append("Be extra careful; multi-vehicle situations are riskier.")
    if sample['Road_Type'].iloc[0] in [0,2]:
        suggestions.append("Pay attention to traffic rules; intersections and dual carriageways are high risk.")
    if not suggestions:
        suggestions.append("Maintain normal caution; conditions seem safe.")

    print("\n💡 Safety Suggestions to Avoid Accidents:")
    for s in suggestions:
        print("-", s)


interact_manual(
    predict_and_visualize,
    Longitude=FloatText(value=-0.1257, description="Longitude"),
    Latitude=FloatText(value=51.5085, description="Latitude"),
    Number_of_Vehicles=IntText(value=2, description="Vehicles"),
    Number_of_Casualties=IntText(value=1, description="Casualties"),
    Day_of_Week=Dropdown(
        options=[
            ("Monday", 1),
            ("Tuesday", 2),
            ("Wednesday", 3),
            ("Thursday", 4),
            ("Friday", 5),
            ("Saturday", 6),
            ("Sunday", 7)
        ],
        description="Day"
    ),
    Weather=Dropdown(options=weather_options, description="Weather"),
    Light_Conditions=Dropdown(options=light_options, description="Light"),
    Road_Type=Dropdown(options=road_type_options, description="Road Type"),
    Speed_limit=IntText(value=30, description="Speed Limit")
)


In [ ]:

df = pd.read_csv("UK_Accident.csv")


severity_labels = {1: "Slight", 2: "Serious", 3: "Fatal"}
df['Accident_Severity_Label'] = df['Accident_Severity'].map(severity_labels)


weather_labels = {0: "Clear", 1: "Rainy", 2: "Snow", 3: "Fog", 4: "Other"}
df['Weather_Label'] = df['Weather_Conditions'].map(weather_labels)


road_labels = {0:"Roundabout",1:"One way",2:"Dual carriageway",3:"Single carriageway",4:"Slip road",5:"Other"}
df['Road_Label'] = df['Road_Type'].map(road_labels)


light_labels = {0: "Daylight", 1: "Dark – lights lit", 2: "Dark – lights unlit", 3: "Other"}
df['Light_Label'] = df['Light_Conditions'].map(light_labels)


df['Hour'] = pd.to_datetime(df['Time'], errors='coerce').dt.hour


df_graph = df.dropna(subset=['Accident_Severity_Label','Weather_Label','Road_Label','Light_Label','Hour'])


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


plt.figure(figsize=(6,4))
sns.countplot(x='Accident_Severity_Label', data=df_graph, palette='coolwarm')
plt.title('Distribution of Accident Severity')
plt.xlabel('Severity')
plt.ylabel('Number of Accidents')
plt.show()


plt.figure(figsize=(8,5))
sns.countplot(x='Weather_Label', hue='Accident_Severity_Label', data=df_graph, palette='Set2')
plt.title('Accident Severity by Weather')
plt.xlabel('Weather')
plt.ylabel('Number of Accidents')
plt.legend(title='Severity')
plt.show()


plt.figure(figsize=(8,5))
sns.countplot(x='Road_Label', hue='Accident_Severity_Label', data=df_graph, palette='Set1')
plt.title('Accident Severity by Road Type')
plt.xlabel('Road Type')
plt.ylabel('Number of Accidents')
plt.legend(title='Severity')
plt.show()


plt.figure(figsize=(10,5))
sns.boxplot(x='Hour', y='Accident_Severity', data=df_graph, palette='cool')
plt.title('Accident Severity by Hour')
plt.xlabel('Hour of Day')
plt.ylabel('Severity')
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


plt.figure(figsize=(6,4))
sns.countplot(x='Accident_Severity_Label', data=df_graph, palette='coolwarm')
plt.title('Distribution of Accident Severity')
plt.xlabel('Severity')
plt.ylabel('Number of Accidents')
plt.show()


plt.figure(figsize=(8,5))
sns.countplot(x='Weather_Label', hue='Accident_Severity_Label', data=df_graph, palette='Set2')
plt.title('Accident Severity by Weather')
plt.xlabel('Weather')
plt.ylabel('Number of Accidents')
plt.legend(title='Severity')
plt.show()


plt.figure(figsize=(8,5))
sns.countplot(x='Road_Label', hue='Accident_Severity_Label', data=df_graph, palette='Set1')
plt.title('Accident Severity by Road Type')
plt.xlabel('Road Type')
plt.ylabel('Number of Accidents')
plt.legend(title='Severity')
plt.show()


plt.figure(figsize=(10,5))
sns.boxplot(x='Hour', y='Accident_Severity', data=df_graph, palette='cool')
plt.title('Accident Severity by Hour')
plt.xlabel('Hour of Day')
plt.ylabel('Severity')
plt.show()


In [ ]:
from ipywidgets import interact_manual, FloatText, IntText, Dropdown


weather_options = ["Clear", "Rainy", "Snow", "Fog", "Other"]
light_options = ["Daylight", "Dark – lights lit", "Dark – lights unlit", "Other"]
road_type_options = ["Roundabout", "One way", "Dual carriageway", "Single carriageway", "Slip road", "Other"]


interact_manual(
    predict_and_visualize,
    Longitude=FloatText(value=-0.1257, description="Longitude"),
    Latitude=FloatText(value=51.5085, description="Latitude"),
    Number_of_Vehicles=IntText(value=2, description="Vehicles"),
    Number_of_Casualties=IntText(value=1, description="Casualties"),
    Day_of_Week=Dropdown(
        options=[
            ("Monday", 1),
            ("Tuesday", 2),
            ("Wednesday", 3),
            ("Thursday", 4),
            ("Friday", 5),
            ("Saturday", 6),
            ("Sunday", 7)
        ],
        description="Day"
    ),
    Weather=Dropdown(options=weather_options, description="Weather"),
    Light_Conditions=Dropdown(options=light_options, description="Light"),
    Road_Type=Dropdown(options=road_type_options, description="Road Type"),
    Speed_limit=IntText(value=30, description="Speed Limit")
)


In [ ]:
def predict_and_visualize(
    Longitude: float,
    Latitude: float,
    Number_of_Vehicles: int,
    Number_of_Casualties: int,
    Day_of_Week: int,
    Weather: str,
    Light_Conditions: str,
    Road_Type: str,
    Speed_limit: int
):

    sample = pd.DataFrame(columns=X.columns)
    sample.loc[0] = 0

    sample.at[0,'Longitude'] = Longitude
    sample.at[0,'Latitude'] = Latitude
    sample.at[0,'Number_of_Vehicles'] = Number_of_Vehicles
    sample.at[0,'Number_of_Casualties'] = Number_of_Casualties
    sample.at[0,'Day_of_Week'] = Day_of_Week
    sample.at[0,'Weather_Conditions'] = list(weather_labels.keys())[weather_options.index(Weather)]
    sample.at[0,'Light_Conditions'] = list(light_labels.keys())[light_options.index(Light_Conditions)]
    sample.at[0,'Road_Type'] = list(road_labels.keys())[road_type_options.index(Road_Type)]
    sample.at[0,'Speed_limit'] = Speed_limit


    sample_scaled = scaler.transform(sample)
    pred = model.predict(sample_scaled)[0]


    print(f"\n✅ Predicted Accident Severity: {severity_labels[pred]}\n")


    suggestions = []
    if pred == 3:
        suggestions.append("⚠️ High risk detected! Take maximum caution; avoid risky maneuvers.")
    elif pred == 2:
        suggestions.append("Be careful; conditions may lead to serious accidents.")
    else:
        suggestions.append("Maintain normal caution; conditions seem safe.")


    weather_value = sample.at[0,'Weather_Conditions']
    light_value = sample.at[0,'Light_Conditions']
    road_value = sample.at[0,'Road_Type']
    speed_value = sample.at[0,'Speed_limit']
    vehicles_value = sample.at[0,'Number_of_Vehicles']

    if weather_value == 1:
        suggestions.append("Drive slowly and keep safe distance; rainy roads are slippery.")
    if weather_value == 2:
        suggestions.append("Use snow tires and drive cautiously; snow increases risk.")
    if light_value != 0:
        suggestions.append("Ensure headlights are on and reduce speed at night or low visibility.")
    if speed_value > 40:
        suggestions.append("Reduce speed; higher speed increases accident severity.")
    if vehicles_value > 2:
        suggestions.append("Be extra careful; multi-vehicle situations are riskier.")
    if road_value in [0,2]:
        suggestions.append("Pay attention to traffic rules; intersections and dual carriageways are high risk.")

    print("\n💡 Safety Suggestions to Avoid Accidents:")
    for s in suggestions:
        print("-", s)


In [ ]:
interact_manual(
    predict_and_visualize,
    Longitude=FloatText(value=-0.1257, description="Longitude"),
    Latitude=FloatText(value=51.5085, description="Latitude"),
    Number_of_Vehicles=IntText(value=2, description="Vehicles"),
    Number_of_Casualties=IntText(value=1, description="Casualties"),
    Day_of_Week=Dropdown(
        options=[("Monday",1),("Tuesday",2),("Wednesday",3),
                 ("Thursday",4),("Friday",5),("Saturday",6),("Sunday",7)],
        description="Day"
    ),
    Weather=Dropdown(options=weather_options, description="Weather"),
    Light_Conditions=Dropdown(options=light_options, description="Light"),
    Road_Type=Dropdown(options=road_type_options, description="Road Type"),
    Speed_limit=IntText(value=30, description="Speed Limit")
)


In [ ]:
def predict_and_visualize(
    Longitude: float,
    Latitude: float,
    Number_of_Vehicles: int,
    Number_of_Casualties: int,
    Day_of_Week: int,
    Weather: str,
    Light_Conditions: str,
    Road_Type: str,
    Speed_limit: int
):

    sample = pd.DataFrame(columns=X.columns)
    sample.loc[0] = 0
    sample.at[0,'Longitude'] = Longitude
    sample.at[0,'Latitude'] = Latitude
    sample.at[0,'Number_of_Vehicles'] = Number_of_Vehicles
    sample.at[0,'Number_of_Casualties'] = Number_of_Casualties
    sample.at[0,'Day_of_Week'] = Day_of_Week
    sample.at[0,'Weather_Conditions'] = list(weather_labels.keys())[weather_options.index(Weather)]
    sample.at[0,'Light_Conditions'] = list(light_labels.keys())[light_options.index(Light_Conditions)]
    sample.at[0,'Road_Type'] = list(road_labels.keys())[road_type_options.index(Road_Type)]
    sample.at[0,'Speed_limit'] = Speed_limit

    sample_scaled = scaler.transform(sample)
    pred = model.predict(sample_scaled)[0]


    from IPython.display import display, Markdown
    display(Markdown(f"## ✅ Predicted Accident Severity: **{severity_labels[pred]}**"))


    suggestions = []
    if pred == 3:
        suggestions.append("⚠️ High risk detected! Take maximum caution; avoid risky maneuvers.")
    elif pred == 2:
        suggestions.append("Be careful; conditions may lead to serious accidents.")
    else:
        suggestions.append("Maintain normal caution; conditions seem safe.")


    weather_value = sample.at[0,'Weather_Conditions']
    light_value = sample.at[0,'Light_Conditions']
    road_value = sample.at[0,'Road_Type']
    speed_value = sample.at[0,'Speed_limit']
    vehicles_value = sample.at[0,'Number_of_Vehicles']

    if weather_value == 1:
        suggestions.append("Drive slowly and keep safe distance; rainy roads are slippery.")
    if weather_value == 2:
        suggestions.append("Use snow tires and drive cautiously; snow increases risk.")
    if light_value != 0:
        suggestions.append("Ensure headlights are on and reduce speed at night or low visibility.")
    if speed_value > 40:
        suggestions.append("Reduce speed; higher speed increases accident severity.")
    if vehicles_value > 2:
        suggestions.append("Be extra careful; multi-vehicle situations are riskier.")
    if road_value in [0,2]:
        suggestions.append("Pay attention to traffic rules; intersections and dual carriageways are high risk.")

    display(Markdown("## 💡 Safety Suggestions to Avoid Accidents"))
    for s in suggestions:
        display(Markdown(f"- {s}"))


    import matplotlib.pyplot as plt
    import seaborn as sns
    sns.set(style="whitegrid")


    plt.figure(figsize=(6,4))
    sns.countplot(x='Accident_Severity_Label', data=df_graph, palette='coolwarm')
    plt.title('Distribution of Accident Severity')
    plt.xlabel('Severity')
    plt.ylabel('Number of Accidents')
    plt.show()


    plt.figure(figsize=(8,5))
    sns.countplot(x='Weather_Label', hue='Accident_Severity_Label', data=df_graph, palette='Set2')
    plt.title('Accident Severity by Weather')
    plt.xlabel('Weather')
    plt.ylabel('Number of Accidents')
    plt.legend(title='Severity')
    plt.show()


    plt.figure(figsize=(8,5))
    sns.countplot(x='Road_Label', hue='Accident_Severity_Label', data=df_graph, palette='Set1')
    plt.title('Accident Severity by Road Type')
    plt.xlabel('Road Type')
    plt.ylabel('Number of Accidents')
    plt.legend(title='Severity')
    plt.show()


    plt.figure(figsize=(10,5))
    sns.boxplot(x='Hour', y='Accident_Severity', data=df_graph, palette='cool')
    plt.title('Accident Severity by Hour')
    plt.xlabel('Hour of Day')
    plt.ylabel('Severity')
    plt.show()


In [ ]:
interact_manual(
    predict_and_visualize,
    Longitude=FloatText(value=-0.1257, description="Longitude"),
    Latitude=FloatText(value=51.5085, description="Latitude"),
    Number_of_Vehicles=IntText(value=2, description="Vehicles"),
    Number_of_Casualties=IntText(value=1, description="Casualties"),
    Day_of_Week=Dropdown(
        options=[("Monday",1),("Tuesday",2),("Wednesday",3),
                 ("Thursday",4),("Friday",5),("Saturday",6),("Sunday",7)],
        description="Day"
    ),
    Weather=Dropdown(options=weather_options, description="Weather"),
    Light_Conditions=Dropdown(options=light_options, description="Light"),
    Road_Type=Dropdown(options=road_type_options, description="Road Type"),
    Speed_limit=IntText(value=30, description="Speed Limit")
)
